<h1 style="text-align: center;">TÉCNICO EM CIÊNCIA DE DADOS</h1>
<h1 style="text-align: center;">Tratando e transformando dados no PySpark</h1>
<br>

**Componente:** Fundamentos de Ambientes e Arquitetura de Dados
<br>
**Unidade Curricular:** Introdução ao PySpark e Processamento Distribuído
<br>
**Tema da Semana:** Transformações em DataFrames
<br>
**Semana 18 - Aula 3**
<br>
**Atividade:** Limpeza e transformação de dados — Gabarito do professor


## Objetivo prático

Aplicar os conceitos estudados na semana em uma atividade de preparação de dados no PySpark.

Nesta proposta, os estudantes devem:

1. Ler um arquivo CSV com `header` e `inferSchema`.
2. Verificar o schema do DataFrame.
3. Converter tipos com `cast()`.
4. Comparar `dropna()` e `fillna()`.
5. Substituir valores com `replace()`.
6. Criar novas colunas com `withColumn()`.
7. Renomear colunas com `withColumnRenamed()`.
8. Adicionar coluna fixa com `lit()`.


## Lista de materiais

- Ambiente com PySpark configurado.
- Este notebook.
- Arquivo auxiliar no mesmo diretório: `DADOSANO2C6B3S18A3_auxiliar.csv`.


## Comentário ao professor

A atividade foi organizada para retomar, em uma única prática, os principais procedimentos trabalhados nas duas aulas teóricas.

O foco é fazer o estudante:
- ler dados;
- observar problemas comuns;
- decidir como tratar nulos;
- padronizar valores;
- criar novas colunas úteis para análise.


### Etapa 1: Ambiente pronto

A SparkSession já está criada para você focar na atividade.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit

spark = (
    SparkSession.builder
    .appName("S18A3_Atividade_Aluno")
    .master("local[*]")
    .getOrCreate()
)

csv_path = "DADOSANO2C6B3S18A3_auxiliar.csv"

print("SparkSession ativa")


SparkSession ativa


### Etapa 2: Leitura do arquivo

Leitura com `header=True` e `inferSchema=True`.

Comentário:
O estudante deve perceber que essas opções ajudam o Spark a interpretar corretamente nomes de colunas e tipos de dados.



In [2]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(csv_path)
)

df.show(8)
df.printSchema()



+---------+--------+--------+-----------+----------+--------------+------+--------------+
|id_pedido| cliente| produto|  categoria|quantidade|preco_unitario|regiao|status_entrega|
+---------+--------+--------+-----------+----------+--------------+------+--------------+
|        1|     Ana| Teclado|Periféricos|         2|           120| Norte|      entregue|
|        2|   Bruno|   Mouse|Periféricos|         5|            60|   Sul|          pend|
|        3|   Carla| Monitor|   Hardware|         1|           900| Leste|      entregue|
|        4|   Diego| Headset|      Áudio|         3|           150| Oeste|      entregue|
|        5|   Elisa|Notebook|   Hardware|         1|          3500| Norte|          pend|
|        6|  Felipe|  Webcam|Periféricos|         4|           210|   Sul|      entregue|
|        7|Gabriela| Cadeira|     Móveis|         1|           980| Leste|      entregue|
|        8|Henrique|    Mesa|     Móveis|         1|          1250| Oeste|          pend|
+---------

### Etapa 3: Ajustando tipos com cast()

Comentário:
Mesmo que o Spark já reconheça alguns tipos corretamente, a etapa reforça o uso de `cast()` como procedimento de preparação e padronização.



In [3]:
df = df.withColumn("quantidade", col("quantidade").cast("int"))
df = df.withColumn("preco_unitario", col("preco_unitario").cast("double"))

df.printSchema()


root
 |-- id_pedido: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- preco_unitario: double (nullable = true)
 |-- regiao: string (nullable = true)
 |-- status_entrega: string (nullable = true)



### Etapa 4: Comparando estratégias para valores nulos

Comentário:
Aqui o estudante deve perceber que `dropna()` remove linhas com valores ausentes, enquanto `fillna()` permite manter os registros.



In [4]:
df_sem_nulos = df.dropna()

print("Quantidade de linhas no DataFrame original:", df.count())
print("Quantidade de linhas no DataFrame com dropna():", df_sem_nulos.count())


Quantidade de linhas no DataFrame original: 50
Quantidade de linhas no DataFrame com dropna(): 42


### Etapa 5: Preenchendo nulos e substituindo valores

Comentário:
Nesta base, a decisão mais adequada é manter os registros, preenchendo nulos com 0 e padronizando valores textuais.



In [5]:
df = df.fillna({
    "quantidade": 0,
    "preco_unitario": 0
})

df = df.replace("indefinida", "não informada", subset=["regiao"])
df = df.replace("pend", "pendente", subset=["status_entrega"])

df.show(10)


+---------+--------+----------+-----------+----------+--------------+-------------+--------------+
|id_pedido| cliente|   produto|  categoria|quantidade|preco_unitario|       regiao|status_entrega|
+---------+--------+----------+-----------+----------+--------------+-------------+--------------+
|        1|     Ana|   Teclado|Periféricos|         2|         120.0|        Norte|      entregue|
|        2|   Bruno|     Mouse|Periféricos|         5|          60.0|          Sul|      pendente|
|        3|   Carla|   Monitor|   Hardware|         1|         900.0|        Leste|      entregue|
|        4|   Diego|   Headset|      Áudio|         3|         150.0|        Oeste|      entregue|
|        5|   Elisa|  Notebook|   Hardware|         1|        3500.0|        Norte|      pendente|
|        6|  Felipe|    Webcam|Periféricos|         4|         210.0|          Sul|      entregue|
|        7|Gabriela|   Cadeira|     Móveis|         1|         980.0|        Leste|      entregue|
|        8

### Etapa 6: Criando e reorganizando colunas

Comentário:
O estudante deve mobilizar três operações centrais da semana:
- `withColumn()` para criar nova informação;
- `withColumnRenamed()` para melhorar a organização;
- `lit()` para registrar uma informação fixa.



In [6]:
df = df.withColumn(
    "valor_total_pedido",
    col("quantidade") * col("preco_unitario")
)

df = df.withColumnRenamed("cliente", "nome_cliente")

df = df.withColumn("origem_dados", lit("csv_pedidos"))

df.show(10, truncate=False)
df.printSchema()


+---------+------------+----------+-----------+----------+--------------+-------------+--------------+------------------+------------+
|id_pedido|nome_cliente|produto   |categoria  |quantidade|preco_unitario|regiao       |status_entrega|valor_total_pedido|origem_dados|
+---------+------------+----------+-----------+----------+--------------+-------------+--------------+------------------+------------+
|1        |Ana         |Teclado   |Periféricos|2         |120.0         |Norte        |entregue      |240.0             |csv_pedidos |
|2        |Bruno       |Mouse     |Periféricos|5         |60.0          |Sul          |pendente      |300.0             |csv_pedidos |
|3        |Carla       |Monitor   |Hardware   |1         |900.0         |Leste        |entregue      |900.0             |csv_pedidos |
|4        |Diego       |Headset   |Áudio      |3         |150.0         |Oeste        |entregue      |450.0             |csv_pedidos |
|5        |Elisa       |Notebook  |Hardware   |1       

## Interpretação (responda no notebook)

1. Qual foi a diferença entre usar `dropna()` e usar `fillna()` neste conjunto de dados?
2. Por que foi importante substituir os valores `"indefinida"` e `"pend"`?
3. Qual é a utilidade da coluna `valor_total_pedido`?
4. O que mudou no DataFrame após renomear a coluna `cliente`?
5. Em que situação uma coluna fixa como `origem_dados` pode ser útil?


## Gabarito das respostas

1. `dropna()` remove registros com valores ausentes, enquanto `fillna()` mantém os registros e substitui os nulos por outro valor.
2. A substituição foi importante para padronizar as informações e deixar o conjunto de dados mais claro para análise.
3. A coluna `valor_total_pedido` mostra o valor total de cada pedido e facilita análises financeiras.
4. O nome da coluna ficou mais claro e organizado, sem alterar os dados armazenados.
5. Uma coluna fixa como `origem_dados` pode ajudar a identificar de qual arquivo, sistema ou etapa os dados vieram.

